In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import img_to_array, load_img
import joblib

In [2]:
#load the models
cnn_model =  joblib.load('MODELS/MRI_model.sav') #CNN model
InseptionV3 = load_model('MODELS/InceptionV3.h5') #InseptionV3
Xception = load_model('MODELS/Xception.h5') #Xception

In [3]:
# Load the Ensemble models to feture extraction
cnn_model_feature =  load_model('MODELS/cnn_ensemble.h5') #CNN model
InseptionV3_feature = load_model('MODELS/inc_ensemble.h5') #InseptionV3
Xception_feature = load_model('MODELS/xcp_ensemble.h5') #Xception

In [4]:
# Load the PKL files ensamble models
cnn_ensemble = joblib.load('MODELS/cnn_ensemble_model.pkl')
inc_ensemble = joblib.load('MODELS/inc_ensemble_model.pkl')
xcp_ensemble = joblib.load('MODELS/xcp_ensemble_model.pkl')
ensemble_model = joblib.load('MODELS/ensemble_model.pkl')

In [5]:
# Create a feature extraction models
dense_layer_name_cnn = 'dense'  
feature_extractor_cnn = Model(inputs=cnn_model_feature.input,
                          outputs=cnn_model_feature.get_layer(dense_layer_name_cnn).output)

dense_layer_name_inc = 'dense_2'  
feature_extractor_inc = Model(inputs=InseptionV3_feature.input,
                          outputs=InseptionV3_feature.get_layer(dense_layer_name_inc).output)

dense_layer_name_xcp = 'dense_3' 
feature_extractor_xcp = Model(inputs=Xception_feature.input,
                          outputs=Xception_feature.get_layer(dense_layer_name_xcp).output)

In [6]:
def resize_image(image_path, size):
    img = load_img(image_path, target_size=size)
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    return img


In [7]:
def extract_features(model, image_path, size):
    img = resize_image(image_path, size)
    features = model.predict(img)
    return features


In [8]:
def predict_with_traditional_model(model, features):
    prediction = model.predict(features)
    #print(f"Prediction shape: {prediction.shape}")  # Debug: print the shape of the prediction array
    #print(f"Prediction values: {prediction}")  # Debug: print the prediction values
    if prediction.ndim == 1:
        return prediction[0]  # Directly return the single prediction if 1-dimensional
    return np.argmax(prediction, axis=1)[0]  # Adjust if your models have different output formats

In [25]:
def predict_with_cnn_model(model, image_path, size):
    img = resize_image(image_path, size)
    prediction = model.predict(img)
    #print(f"model prediction: {prediction}")  # Print the CNN model prediction
    return np.argmax(prediction, axis=1)[0]


In [26]:
def majority_vote(predictions):
    return max(set(predictions), key=predictions.count)

In [27]:
def make_predictions(image_path):
    predictions = []

    cnn_pred = predict_with_cnn_model(cnn_model, image_path, (256, 256))
    inc_pred = predict_with_cnn_model(InseptionV3, image_path, (256, 256))
    xcp_pred = predict_with_cnn_model(Xception, image_path, (256, 256))

    # CNN models predictions
    predictions.append(cnn_pred)
    predictions.append(inc_pred)
    predictions.append(xcp_pred)

    #print(f"CNN model prediction: {cnn_pred}")
    #print(f"InceptionV3 model prediction: {inc_pred}")
    #print(f"Xception model prediction: {xcp_pred}")

    # Extract features using the feature extraction model
    cnn_features = extract_features(feature_extractor_cnn, image_path, (128, 128))
    inc_features = extract_features(feature_extractor_inc, image_path, (128, 128))
    exc_features = extract_features(feature_extractor_xcp, image_path, (128, 128))

    # Traditional ensemble models predictions
    cnn_en_prediction = predict_with_traditional_model(cnn_ensemble, cnn_features)
    inc_en_prediction = predict_with_traditional_model(inc_ensemble, inc_features)
    exc_en_prediction = predict_with_traditional_model(xcp_ensemble, exc_features)

    #print(f"Traditional CNN ensemble model prediction: {cnn_en_prediction}")
    #print(f"Traditional InceptionV3 ensemble model prediction: {inc_en_prediction}")
    #print(f"Traditional Xception ensemble model prediction: {exc_en_prediction}")
    

    predictions.append(cnn_en_prediction)
    predictions.append(inc_en_prediction)
    predictions.append(exc_en_prediction)

    # Perform majority voting
    final_prediction = majority_vote(predictions)
    

    return final_prediction

In [28]:
# Example usage
image_path = 'meningioma.jpg'
final_prediction = make_predictions(image_path)
print(f"Final prediction: {final_prediction}")

1/1 [==============================] - 0s 113ms/step
Final prediction: 1


In [33]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score

In [34]:
def load_test_data(directory, target_size, batch_size=32):
    datagen = ImageDataGenerator(rescale=1./255)
    generator = datagen.flow_from_directory(
        directory,
        target_size=target_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False  # Do not shuffle to keep track of file names
    )
    return generator


In [47]:
def evaluate_model_on_dataset(test_directory, target_size):
    # Load the test data
    test_generator = load_test_data(test_directory, target_size)
    
    y_true = []
    y_pred = []

    for i in range(len(test_generator)):
        x_batch, y_batch = test_generator[i]
        for j in range(len(x_batch)):
            # Get the file path
            image_path = test_generator.filepaths[i * test_generator.batch_size + j]
            # Get the true label
            true_label = np.argmax(y_batch[j])
            # Get the prediction
            pred_label = make_predictions(image_path)
            # Append to the lists
            y_true.append(true_label)
            y_pred.append(pred_label)

            # Print the prediction and the true label
            print(f"Image: {image_path}")
            print(f"True label: {true_label}")
            print(f"Predicted label: {pred_label}")
            print("------")
    
    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Ensemble model accuracy: {accuracy * 100:.2f}%")
    
    return y_true, y_pred

In [48]:
import tqdm

In [60]:
#for i in tqdm(range(len(test_generator)), desc="Processing batches")

def evaluate_model_on_dataset(test_directory, target_size):
    # Load the test data
    test_generator = load_test_data(test_directory, target_size)
    
    y_true = []
    y_pred = []

    for i in tqdm(range(len(test_generator)), desc="Processing batches"):
        x_batch, y_batch = test_generator[i]
        for j in range(len(x_batch)):
            # Get the file path
            image_path = test_generator.filepaths[i * test_generator.batch_size + j]
            # Get the true label
            true_label = np.argmax(y_batch[j])
            # Get the prediction
            pred_label = make_predictions(image_path)
            # Append to the lists
            y_true.append(true_label)
            y_pred.append(pred_label)

            # Print the prediction and the true label
            print(f"Image: {image_path}")
            print(f"True label: {true_label}")
            print(f"Predicted label: {pred_label}")
            print("------")
    
    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Ensemble model accuracy: {accuracy * 100:.2f}%")
    
    return y_true, y_pred

In [ ]:

test_directory = 'Testing'  # Update this to the path of your test data
y_true, y_pred = evaluate_model_on_dataset(test_directory, target_size=(256, 256))

Found 6985 images belonging to 4 classes.
1/1 [==============================] - 0s 203ms/step
Image: Testing\glioma\G_659_HF_.jpg
True label: 0
Predicted label: 0
------
1/1 [==============================] - 0s 102ms/step
Image: Testing\glioma\G_659_RO_.jpg
True label: 0
Predicted label: 0
------
1/1 [==============================] - 0s 88ms/step
Image: Testing\glioma\G_659_SP_.jpg
True label: 0
Predicted label: 0
------
1/1 [==============================] - 0s 103ms/step
Image: Testing\glioma\G_659_VF_.jpg
True label: 0
Predicted label: 0
------
1/1 [==============================] - 0s 83ms/step
Image: Testing\glioma\G_660.jpg
True label: 0
Predicted label: 0
------
1/1 [==============================] - 0s 82ms/step
Image: Testing\glioma\G_660_BR_.jpg
True label: 0
Predicted label: 0
------
1/1 [==============================] - 0s 93ms/step
Image: Testing\glioma\G_660_DA_.jpg
True label: 0
Predicted label: 0
------
1/1 [==============================] - 0s 95ms/step
Image: Test

In [59]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Create the confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# Plot the confusion matrix
#plt.figure(figsize=(10, 7))
#sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
#plt.xlabel('Predicted')
#plt.ylabel('True')
#plt.title('Confusion Matrix')
#plt.show()
print("------------------------------------------------------------------")
# Print classification report
print("Classification Report:")
print(classification_report(y_true, y_pred))

print("------------------------------------------------------------------")
accuracy = accuracy_score(y_true, y_pred)
print(f"Ensemble model accuracy: {accuracy * 100:.2f}%")

Confusion Matrix:
[[1688   34    6    5]
 [  42 1636    8   13]
 [   5    5 1837    0]
 [   4    4    0 1698]]
------------------------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1733
           1       0.97      0.96      0.97      1699
           2       0.99      0.99      0.99      1847
           3       0.99      1.00      0.99      1706

    accuracy                           0.98      6985
   macro avg       0.98      0.98      0.98      6985
weighted avg       0.98      0.98      0.98      6985

------------------------------------------------------------------
Ensemble model accuracy: 98.20%
